# Perbandingan SVM vs Random Forest vs Logistic Regression vs BERT vs Hybrid SVM

Notebook ini membandingkan lima pendekatan klasifikasi tiket IT helpdesk:

| Pendekatan | Deskripsi |
|---|---|
| **SVM** | TF-IDF (unigram+bigram) + LinearSVC |
| **Random Forest** | TF-IDF (unigram+bigram) + RandomForestClassifier |
| **Logistic Regression** | TF-IDF (unigram+bigram) + LogisticRegression |
| **BERT** | Fine-tuned DistilBERT multilingual |
| **Hybrid SVM** | SVM sebagai dasar, GenAI koreksi mismatch SVM |

**Input:** `cobacek.xlsx`  
**Output:** `cobacek_compare.xlsx`

> **Catatan waktu:** SVM, RF, dan LR selesai dalam hitungan detik. BERT membutuhkan beberapa menit di CPU (lebih cepat dengan GPU).

## 1. Install Dependencies

In [1]:
# !pip install pandas scikit-learn openpyxl openai python-dotenv torch transformers

## 2. Import Libraries

In [2]:
import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path("../src").resolve()))

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from bert_classifier import BertClassifier

load_dotenv()
print("Libraries loaded.")

Libraries loaded.


## 3. Konfigurasi

In [ ]:
# ===================== UBAH SESUAI KEBUTUHAN =====================
INPUT_FILE      = "../data/cobacek20data.xlsx"
OUTPUT_FILE     = "../results/cobacek_compare_20datates.xlsx"
N_SPLITS        = 5     # jumlah fold untuk K-Fold Cross-Validation

# Model GenAI — atur di .env pada OPENAI_MODELS (pisah koma)
_env_models  = [m.strip() for m in os.getenv("OPENAI_MODELS", "").split(",") if m.strip()]
MULTI_MODELS = _env_models
FORCED_MODEL = None

# Logistic Regression — set SKIP_LR = True untuk melewati
SKIP_LR         = False

# BERT — set SKIP_BERT = True untuk melewati BERT (lebih cepat)
SKIP_BERT       = False
BERT_MODEL_NAME = "distilbert-base-multilingual-cased"
BERT_EPOCHS     = 2     # 3 epoch cukup untuk fine-tuning, 2 lebih cepat ~1.5x
BERT_BATCH_SIZE = 32    # default 16; 32 mengurangi jumlah step ~1.5x
BERT_MAX_LENGTH = 64    # default 128; tiket helpdesk biasanya <50 token, 64 cukup
BERT_N_SPLITS   = 3     # fold khusus BERT (lebih sedikit dari N_SPLITS agar lebih cepat)
# =================================================================

print(f"Model yang akan dipakai: {MULTI_MODELS}")
print(f"K-Fold: {N_SPLITS} folds | BERT K-Fold: {BERT_N_SPLITS} folds")

## 4. Helper Functions

In [4]:
def safe_str(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value)


def get_available_models(client: OpenAI) -> List[str]:
    models = client.models.list()
    return sorted({m.id for m in models.data})


def build_svm_pipeline() -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("svm", LinearSVC()),
    ])


def build_rf_pipeline() -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ])


def build_lr_pipeline() -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("lr", LogisticRegression(max_iter=1000, random_state=42)),
    ])


def classify_with_genai(
    client: OpenAI,
    model: str,
    text: str,
    allowed_categories: List[str],
    allowed_priorities: List[str],
    ml_category: Optional[str] = None,
    ml_priority: Optional[str] = None,
    retries: int = 2,
) -> Tuple[str, str]:
    default_category = ml_category or allowed_categories[0]
    default_priority = ml_priority or allowed_priorities[0]

    prompt = f"""You are an IT helpdesk ticket classifier.
Classify the ticket below into the correct category and priority based on its content.
Return strict JSON only with keys: category, priority. No explanation, no markdown.

Allowed category values:
{json.dumps(allowed_categories, ensure_ascii=True)}

Allowed priority values:
{json.dumps(allowed_priorities, ensure_ascii=True)}

SVM predicted: category='{ml_category}', priority='{ml_priority}'. Use as a hint only — re-classify independently based on the text.

Ticket:
\"\"\"{text[:3000]}\"\"\"
""".strip()

    for attempt in range(retries):
        try:
            response = client.responses.create(model=model, input=prompt)
            raw = (response.output_text or "").strip()
            parsed = json.loads(raw)
            category = str(parsed.get("category", default_category)).strip()
            priority = str(parsed.get("priority", default_priority)).strip()
            if category not in allowed_categories:
                category = default_category
            if priority not in allowed_priorities:
                priority = default_priority
            return category, priority
        except Exception:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                return default_category, default_priority

    return default_category, default_priority


def metrics_dict(y_true: List[str], y_pred: List[str], label_name: str) -> Dict[str, float]:
    report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    return {
        "label": label_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "samples": len(y_true),
    }


print("Helper functions defined.")

Helper functions defined.


## 5. Load Data

In [5]:
df = pd.read_excel(INPUT_FILE)
required = ["description", "category", "priority"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ada: {missing}")

df["description"] = df["description"].apply(safe_str)
df["category"]    = df["category"].apply(safe_str)
df["priority"]    = df["priority"].apply(safe_str)

print(f"Total baris: {len(df)}")
print(f"\nDistribusi category:\n{df['category'].value_counts()}")
print(f"\nDistribusi priority:\n{df['priority'].value_counts()}")

Total baris: 20

Distribusi category:
category
Product       5
Network       3
Account       2
Billing       2
Feature       2
Bug           2
Outage        1
Inquiry       1
Marketing     1
Disruption    1
Name: count, dtype: int64

Distribusi priority:
priority
medium    11
high       8
low        1
Name: count, dtype: int64


## 6. K-Fold Cross-Validation Setup

> Semua model dievaluasi dengan **Stratified K-Fold (5 fold)**.
> Setiap baris diuji tepat sekali pada data yang belum pernah dilihat model — tidak ada data yang terbuang.


In [6]:
lr_available = not SKIP_LR

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for col in ["svm_category", "svm_priority", "rf_category", "rf_priority"]:
    df[col] = ""
if lr_available:
    for col in ["lr_category", "lr_priority"]:
        df[col] = ""

print(f"K-Fold: {N_SPLITS} folds | {len(df)} baris total")
lbl = "+LR" if lr_available else ""
print(f"Training SVM + RF{lbl} per fold...")

svm_time = 0.0
rf_time  = 0.0
lr_time  = 0.0

for fold, (train_idx, test_idx) in enumerate(kf.split(df["description"], df["category"]), 1):
    train_desc = df["description"].iloc[train_idx]
    train_cat  = df["category"].iloc[train_idx]
    train_pri  = df["priority"].iloc[train_idx]
    test_desc  = df["description"].iloc[test_idx]
    test_index = df.index[test_idx]

    # SVM
    t0 = time.time()
    svm_c = build_svm_pipeline(); svm_p = build_svm_pipeline()
    svm_c.fit(train_desc, train_cat); svm_p.fit(train_desc, train_pri)
    df.loc[test_index, "svm_category"] = svm_c.predict(test_desc)
    df.loc[test_index, "svm_priority"]  = svm_p.predict(test_desc)
    svm_time += time.time() - t0

    # Random Forest
    t0 = time.time()
    rf_c = build_rf_pipeline(); rf_p = build_rf_pipeline()
    rf_c.fit(train_desc, train_cat); rf_p.fit(train_desc, train_pri)
    df.loc[test_index, "rf_category"] = rf_c.predict(test_desc)
    df.loc[test_index, "rf_priority"]  = rf_p.predict(test_desc)
    rf_time += time.time() - t0

    # Logistic Regression
    if lr_available:
        t0 = time.time()
        lr_c = build_lr_pipeline(); lr_p = build_lr_pipeline()
        lr_c.fit(train_desc, train_cat); lr_p.fit(train_desc, train_pri)
        df.loc[test_index, "lr_category"] = lr_c.predict(test_desc)
        df.loc[test_index, "lr_priority"]  = lr_p.predict(test_desc)
        lr_time += time.time() - t0

    print(f"  Fold {fold}/{N_SPLITS} selesai")

df["svm_match"] = (df["svm_category"] == df["category"]) & (df["svm_priority"] == df["priority"])
df["rf_match"]  = (df["rf_category"]  == df["category"]) & (df["rf_priority"]  == df["priority"])
if lr_available:
    df["lr_match"] = (df["lr_category"] == df["category"]) & (df["lr_priority"] == df["priority"])

print("\nOOF Accuracy:")
print(f"  SVM  Cat: {accuracy_score(df['category'], df['svm_category']):.4f} | Pri: {accuracy_score(df['priority'], df['svm_priority']):.4f} | Waktu: {svm_time:.1f}s")
print(f"  RF   Cat: {accuracy_score(df['category'], df['rf_category']):.4f} | Pri: {accuracy_score(df['priority'], df['rf_priority']):.4f} | Waktu: {rf_time:.1f}s")
if lr_available:
    print(f"  LR   Cat: {accuracy_score(df['category'], df['lr_category']):.4f} | Pri: {accuracy_score(df['priority'], df['lr_priority']):.4f} | Waktu: {lr_time:.1f}s")


K-Fold: 5 folds | 20 baris total
Training SVM + RF+LR per fold...


C:\Users\itsupport\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


  Fold 1/5 selesai
  Fold 2/5 selesai
  Fold 3/5 selesai
  Fold 4/5 selesai
  Fold 5/5 selesai

OOF Accuracy:
  SVM  Cat: 0.4000 | Pri: 0.4500 | Waktu: 0.1s
  RF   Cat: 0.3000 | Pri: 0.5500 | Waktu: 3.0s
  LR   Cat: 0.2500 | Pri: 0.5500 | Waktu: 0.2s


## 7. Training SVM

In [7]:
print("=== SVM — Accuracy (OOF) ===")
print(f"Category : {accuracy_score(df['category'], df['svm_category']):.4f}")
print(f"Priority : {accuracy_score(df['priority'], df['svm_priority']):.4f}")


=== SVM — Accuracy (OOF) ===
Category : 0.4000
Priority : 0.4500


In [8]:
print("=== SVM — Category ===")
print(f"Accuracy : {accuracy_score(df['category'], df['svm_category']):.4f}")
print(classification_report(df["category"], df["svm_category"], zero_division=0))

print("=== SVM — Priority ===")
print(f"Accuracy : {accuracy_score(df['priority'], df['svm_priority']):.4f}")
print(classification_report(df["priority"], df["svm_priority"], zero_division=0))


=== SVM — Category ===
Accuracy : 0.4000
              precision    recall  f1-score   support

     Account       0.00      0.00      0.00         2
     Billing       1.00      0.50      0.67         2
         Bug       0.00      0.00      0.00         2
  Disruption       0.00      0.00      0.00         1
     Feature       0.00      0.00      0.00         2
     Inquiry       0.00      0.00      0.00         1
   Marketing       0.00      0.00      0.00         1
     Network       0.29      0.67      0.40         3
      Outage       0.00      0.00      0.00         1
     Product       0.45      1.00      0.62         5

    accuracy                           0.40        20
   macro avg       0.17      0.22      0.17        20
weighted avg       0.26      0.40      0.28        20

=== SVM — Priority ===
Accuracy : 0.4500
              precision    recall  f1-score   support

        high       0.00      0.00      0.00         8
         low       0.00      0.00      0.00       

## 8. Training Random Forest

In [9]:
print("=== Random Forest — Accuracy (OOF) ===")
print(f"Category : {accuracy_score(df['category'], df['rf_category']):.4f}")
print(f"Priority : {accuracy_score(df['priority'], df['rf_priority']):.4f}")


=== Random Forest — Accuracy (OOF) ===
Category : 0.3000
Priority : 0.5500


In [10]:
print("=== Random Forest — Category ===")
print(f"Accuracy : {accuracy_score(df['category'], df['rf_category']):.4f}")
print(classification_report(df["category"], df["rf_category"], zero_division=0))

print("=== Random Forest — Priority ===")
print(f"Accuracy : {accuracy_score(df['priority'], df['rf_priority']):.4f}")
print(classification_report(df["priority"], df["rf_priority"], zero_division=0))


=== Random Forest — Category ===
Accuracy : 0.3000
              precision    recall  f1-score   support

     Account       0.00      0.00      0.00         2
     Billing       0.00      0.00      0.00         2
         Bug       0.00      0.00      0.00         2
  Disruption       0.00      0.00      0.00         1
     Feature       0.00      0.00      0.00         2
     Inquiry       0.00      0.00      0.00         1
   Marketing       0.00      0.00      0.00         1
     Network       0.25      0.33      0.29         3
      Outage       0.00      0.00      0.00         1
     Product       0.36      1.00      0.53         5

    accuracy                           0.30        20
   macro avg       0.06      0.13      0.08        20
weighted avg       0.13      0.30      0.17        20

=== Random Forest — Priority ===
Accuracy : 0.5500
              precision    recall  f1-score   support

        high       0.00      0.00      0.00         8
         low       0.00      0

## 9. Training Logistic Regression

In [11]:
if lr_available:
    print("=== Logistic Regression — Accuracy (OOF) ===")
    print(f"Category : {accuracy_score(df['category'], df['lr_category']):.4f}")
    print(f"Priority : {accuracy_score(df['priority'], df['lr_priority']):.4f}")
else:
    print("Logistic Regression dilewati (SKIP_LR=True).")


=== Logistic Regression — Accuracy (OOF) ===
Category : 0.2500
Priority : 0.5500


In [12]:
if lr_available:
    print("=== Logistic Regression — Category ===")
    print(f"Accuracy : {accuracy_score(df['category'], df['lr_category']):.4f}")
    print(classification_report(df["category"], df["lr_category"], zero_division=0))

    print("=== Logistic Regression — Priority ===")
    print(f"Accuracy : {accuracy_score(df['priority'], df['lr_priority']):.4f}")
    print(classification_report(df["priority"], df["lr_priority"], zero_division=0))


=== Logistic Regression — Category ===
Accuracy : 0.2500
              precision    recall  f1-score   support

     Account       0.00      0.00      0.00         2
     Billing       0.00      0.00      0.00         2
         Bug       0.00      0.00      0.00         2
  Disruption       0.00      0.00      0.00         1
     Feature       0.00      0.00      0.00         2
     Inquiry       0.00      0.00      0.00         1
   Marketing       0.00      0.00      0.00         1
     Network       0.00      0.00      0.00         3
      Outage       0.00      0.00      0.00         1
     Product       0.28      1.00      0.43         5

    accuracy                           0.25        20
   macro avg       0.03      0.10      0.04        20
weighted avg       0.07      0.25      0.11        20

=== Logistic Regression — Priority ===
Accuracy : 0.5500
              precision    recall  f1-score   support

        high       0.00      0.00      0.00         8
         low      

## 10. Fine-tuning BERT

> Training BERT pada CPU membutuhkan waktu lebih lama. Set `SKIP_BERT = True` di sel konfigurasi untuk melewati.

In [ ]:
bert_available = not SKIP_BERT

if bert_available:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device} | Model: {BERT_MODEL_NAME}")
    print(f"BERT K-Fold: {BERT_N_SPLITS} folds | Epochs: {BERT_EPOCHS} | Batch: {BERT_BATCH_SIZE} | MaxLen: {BERT_MAX_LENGTH}")
    print(f"Training BERT — akan cukup lama di CPU...")

    df["bert_category"] = ""
    df["bert_priority"]  = ""

    kf_bert = StratifiedKFold(n_splits=BERT_N_SPLITS, shuffle=True, random_state=42)

    bert_time = 0.0
    for fold, (train_idx, test_idx) in enumerate(kf_bert.split(df["description"], df["category"]), 1):
        print(f"\n[BERT] Fold {fold}/{BERT_N_SPLITS}...")
        train_desc = df["description"].iloc[train_idx].tolist()
        train_cat  = df["category"].iloc[train_idx].tolist()
        train_pri  = df["priority"].iloc[train_idx].tolist()
        test_desc  = df["description"].iloc[test_idx].tolist()
        test_index = df.index[test_idx]

        t0 = time.time()
        bert_c = BertClassifier(model_name=BERT_MODEL_NAME, epochs=BERT_EPOCHS,
                                batch_size=BERT_BATCH_SIZE, max_length=BERT_MAX_LENGTH)
        bert_p = BertClassifier(model_name=BERT_MODEL_NAME, epochs=BERT_EPOCHS,
                                batch_size=BERT_BATCH_SIZE, max_length=BERT_MAX_LENGTH)
        bert_c.fit(train_desc, train_cat)
        bert_p.fit(train_desc, train_pri)
        df.loc[test_index, "bert_category"] = bert_c.predict(test_desc)
        df.loc[test_index, "bert_priority"]  = bert_p.predict(test_desc)
        bert_time += time.time() - t0
        print(f"  Fold {fold} selesai")

    df["bert_match"] = (
        (df["bert_category"] == df["category"]) &
        (df["bert_priority"]  == df["priority"])
    )
    print(f"\nBERT Category Acc (OOF) : {accuracy_score(df['category'], df['bert_category']):.4f}")
    print(f"BERT Priority  Acc (OOF) : {accuracy_score(df['priority'], df['bert_priority']):.4f}")
    print(f"BERT Total Time          : {bert_time:.1f}s")
else:
    bert_time = 0.0
    print("BERT dilewati (SKIP_BERT=True).")

In [14]:
if bert_available:
    print("=== BERT — Category ===")
    print(f"Accuracy : {accuracy_score(df['category'], df['bert_category']):.4f}")
    print(classification_report(df["category"], df["bert_category"], zero_division=0))

    print("=== BERT — Priority ===")
    print(f"Accuracy : {accuracy_score(df['priority'], df['bert_priority']):.4f}")
    print(classification_report(df["priority"], df["bert_priority"], zero_division=0))


=== BERT — Category ===
Accuracy : 0.0500
              precision    recall  f1-score   support

     Account       0.00      0.00      0.00         2
     Billing       0.00      0.00      0.00         2
         Bug       0.00      0.00      0.00         2
  Disruption       0.00      0.00      0.00         1
     Feature       0.00      0.00      0.00         2
     Inquiry       0.00      0.00      0.00         1
   Marketing       0.00      0.00      0.00         1
     Network       0.00      0.00      0.00         3
      Outage       0.00      0.00      0.00         1
     Product       0.17      0.20      0.18         5

    accuracy                           0.05        20
   macro avg       0.02      0.02      0.02        20
weighted avg       0.04      0.05      0.05        20

=== BERT — Priority ===
Accuracy : 0.5000
              precision    recall  f1-score   support

        high       0.00      0.00      0.00         8
         low       0.00      0.00      0.00     

## 11. Pilih Model GenAI

In [15]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY tidak ditemukan di .env")

client = OpenAI(api_key=api_key)
available_models = get_available_models(client)

if MULTI_MODELS:
    requested = [m.strip() for m in MULTI_MODELS if m.strip()]
elif FORCED_MODEL:
    requested = [FORCED_MODEL]
else:
    raise ValueError("Tidak ada model yang dikonfigurasi. Set MULTI_MODELS atau FORCED_MODEL di sel Konfigurasi.")

# Skip model yang tidak available di akun ini
selected_models = []
for m in requested:
    if m in available_models:
        selected_models.append(m)
    else:
        print(f"[WARNING] Model '{m}' tidak tersedia di akun ini — dilewati.")

if not selected_models:
    raise ValueError(f"Tidak ada model yang tersedia dari daftar: {requested}")

print(f"Model yang digunakan: {selected_models}")

Model yang digunakan: ['gpt-5.4-mini', 'gpt-4.1-mini', 'gpt-4o-mini']


## 12. Tentukan Mismatch SVM

In [16]:
allowed_categories = sorted(df["category"].dropna().unique().tolist())
allowed_priorities  = sorted(df["priority"].dropna().unique().tolist())

# Mismatch berdasarkan OOF SVM predictions
svm_mismatch_mask    = ((df["svm_category"] != df["category"]) | (df["svm_priority"] != df["priority"]))
svm_mismatch_indices = df[svm_mismatch_mask].index

print(f"OOF SVM mismatch : {svm_mismatch_mask.sum()} baris")
print(f"Akan dikoreksi GenAI : {len(svm_mismatch_indices)} baris")


OOF SVM mismatch : 16 baris
Akan dikoreksi GenAI : 16 baris


## 13. Inisialisasi Metrik Baseline

In [17]:
metrics_rows = [
    {"approach": "SVM",           "elapsed_seconds": round(svm_time, 2), **metrics_dict(df["category"].tolist(), df["svm_category"].tolist(), "category")},
    {"approach": "SVM",           "elapsed_seconds": round(svm_time, 2), **metrics_dict(df["priority"].tolist(),  df["svm_priority"].tolist(),  "priority")},
    {"approach": "Random Forest", "elapsed_seconds": round(rf_time,  2), **metrics_dict(df["category"].tolist(), df["rf_category"].tolist(),  "category")},
    {"approach": "Random Forest", "elapsed_seconds": round(rf_time,  2), **metrics_dict(df["priority"].tolist(),  df["rf_priority"].tolist(),   "priority")},
]
if lr_available:
    metrics_rows += [
        {"approach": "Logistic Regression", "elapsed_seconds": round(lr_time, 2), **metrics_dict(df["category"].tolist(), df["lr_category"].tolist(), "category")},
        {"approach": "Logistic Regression", "elapsed_seconds": round(lr_time, 2), **metrics_dict(df["priority"].tolist(),  df["lr_priority"].tolist(),  "priority")},
    ]
if bert_available:
    metrics_rows += [
        {"approach": "BERT", "elapsed_seconds": round(bert_time, 2), **metrics_dict(df["category"].tolist(), df["bert_category"].tolist(), "category")},
        {"approach": "BERT", "elapsed_seconds": round(bert_time, 2), **metrics_dict(df["priority"].tolist(),  df["bert_priority"].tolist(),  "priority")},
    ]
print("Baseline metrics initialized.")


Baseline metrics initialized.


## 14. Jalankan Hybrid SVM per Model

In [18]:
for selected_model in selected_models:
    model_key          = re.sub(r"[^a-zA-Z0-9]+", "_", selected_model).strip("_").lower()
    hybrid_svm_cat_col = f"hybrid_svm_category_{model_key}"
    hybrid_svm_pri_col = f"hybrid_svm_priority_{model_key}"

    df[hybrid_svm_cat_col] = df["svm_category"]
    df[hybrid_svm_pri_col] = df["svm_priority"]

    print(f"\n[Hybrid SVM] {selected_model}")
    hybrid_time = 0.0
    for processed, idx in enumerate(svm_mismatch_indices, start=1):
        t0 = time.time()
        new_cat, new_pri = classify_with_genai(
            client=client, model=selected_model, text=df.at[idx, "description"],
            allowed_categories=allowed_categories, allowed_priorities=allowed_priorities,
            ml_category=df.at[idx, "svm_category"], ml_priority=df.at[idx, "svm_priority"],
        )
        hybrid_time += time.time() - t0
        df.at[idx, hybrid_svm_cat_col] = new_cat
        df.at[idx, hybrid_svm_pri_col] = new_pri
        if processed % 20 == 0:
            print(f"  {processed}/{len(svm_mismatch_indices)} selesai")

    metrics_rows.append({"approach": f"Hybrid SVM ({selected_model})", "elapsed_seconds": round(hybrid_time, 2),
        **metrics_dict(df["category"].tolist(), df[hybrid_svm_cat_col].tolist(), "category")})
    metrics_rows.append({"approach": f"Hybrid SVM ({selected_model})", "elapsed_seconds": round(hybrid_time, 2),
        **metrics_dict(df["priority"].tolist(),  df[hybrid_svm_pri_col].tolist(),  "priority")})

print("\nSelesai.")



[Hybrid SVM] gpt-5.4-mini

[Hybrid SVM] gpt-4.1-mini

[Hybrid SVM] gpt-4o-mini

Selesai.


## 15. Tampilkan Ringkasan Metrik

In [19]:
metrics_df = pd.DataFrame(metrics_rows)

# Semua metrik per label, category & priority berdampingan
metric_vals = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]
pivot = (
    metrics_df[["approach", "label"] + metric_vals + ["samples"]]
    .pivot_table(index="approach", columns="label", values=metric_vals + ["samples"], aggfunc="first")
)
pivot.columns = [f"{val}_{lbl}" for val, lbl in pivot.columns]

# elapsed_seconds sama untuk kedua label — ambil dari baris category
elapsed = metrics_df[metrics_df["label"] == "category"].set_index("approach")["elapsed_seconds"]
pivot.insert(0, "Waktu (s)", elapsed)

ordered = [
    "Waktu (s)",
    "accuracy_category", "accuracy_priority",
    "macro_precision_category", "macro_precision_priority",
    "macro_recall_category", "macro_recall_priority",
    "macro_f1_category", "macro_f1_priority",
    "weighted_f1_category", "weighted_f1_priority",
    "samples_category",
]
pivot = pivot[ordered]
pivot.columns = [
    "Waktu (s)",
    "Acc Cat", "Acc Pri",
    "Prec Cat", "Prec Pri",
    "Rec Cat", "Rec Pri",
    "F1 Cat", "F1 Pri",
    "wF1 Cat", "wF1 Pri",
    "Samples",
]
pivot = pivot.sort_values("Acc Cat", ascending=False)

metric_cols = ["Acc Cat", "Acc Pri", "Prec Cat", "Prec Pri", "Rec Cat", "Rec Pri", "F1 Cat", "F1 Pri", "wF1 Cat", "wF1 Pri"]
print("=" * 60)
print("PERBANDINGAN SEMUA SKEMA")
print("=" * 60)
display(pivot.style
        .format("{:.4f}", subset=metric_cols)
        .format("{:.2f}", subset=["Waktu (s)"])
        .format("{:.0f}", subset=["Samples"])
        .background_gradient(cmap="RdYlGn", subset=metric_cols))


PERBANDINGAN SEMUA SKEMA


,Waktu (s),Acc Cat,Acc Pri,Prec Cat,Prec Pri,Rec Cat,Rec Pri,F1 Cat,F1 Pri,wF1 Cat,wF1 Pri,Samples
approach,,,,,,,,,,,,
Hybrid SVM (gpt-4.1-mini),28.13,0.5500,0.5500,0.5083,0.5569,0.6567,0.6705,0.5105,0.4833,0.5245,0.5975,20
Hybrid SVM (gpt-4o-mini),23.63,0.5500,0.5500,0.5417,0.3571,0.6067,0.3674,0.5171,0.3562,0.5479,0.5234,20
Hybrid SVM (gpt-5.4-mini),17.80,0.4500,0.5500,0.4200,0.5202,0.4567,0.6591,0.4033,0.4589,0.4517,0.5597,20
SVM,0.06,0.4000,0.4500,0.1740,0.1667,0.2167,0.2727,0.1692,0.2069,0.2829,0.3414,20
Random Forest,2.96,0.3000,0.5500,0.0607,0.1833,0.1333,0.3333,0.0812,0.2366,0.1744,0.3903,20
Logistic Regression,0.19,0.2500,0.5500,0.0278,0.1833,0.1000,0.3333,0.0435,0.2366,0.1087,0.3903,20
BERT,184.56,0.0500,0.5000,0.0167,0.1754,0.0200,0.3030,0.0182,0.2222,0.0455,0.3667,20


## 16. Export ke Excel

In [20]:
import openpyxl
from openpyxl.comments import Comment

metrics_df = pd.DataFrame(metrics_rows)

summary_rows = [
    {"key": "selected_models",             "value": ", ".join(selected_models)},
    {"key": "bert_model",                  "value": BERT_MODEL_NAME if bert_available else "skipped"},
    {"key": "total_rows",                  "value": len(df)},
    {"key": "k_folds",                     "value": N_SPLITS},
    {"key": "svm_mismatch_rows_corrected", "value": len(svm_mismatch_indices)},
]
summary_df = pd.DataFrame(summary_rows)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(writer,         index=False, sheet_name="Predictions_Compare")
    metrics_df.to_excel(writer, index=False, sheet_name="Metrics")
    summary_df.to_excel(writer, index=False, sheet_name="Summary")

# ── Keterangan header per sheet ──────────────────────────────────────────────
NOTES_PREDICTIONS = {
    "subject":      "Judul / subjek tiket IT helpdesk",
    "description":  "Deskripsi lengkap tiket — input utama yang digunakan semua model",
    "answer":       "Jawaban atau solusi yang diberikan untuk tiket ini",
    "type":         "Tipe tiket (contoh: Request, Incident)",
    "priority":     "Prioritas AKTUAL (ground truth) dari tiket",
    "category":     "Kategori AKTUAL (ground truth) dari tiket",
    "svm_category": "Prediksi kategori oleh SVM (TF-IDF + LinearSVC, OOF K-Fold)",
    "svm_priority": "Prediksi prioritas oleh SVM (TF-IDF + LinearSVC, OOF K-Fold)",
    "rf_category":  "Prediksi kategori oleh Random Forest (TF-IDF, OOF K-Fold)",
    "rf_priority":  "Prediksi prioritas oleh Random Forest (TF-IDF, OOF K-Fold)",
    "lr_category":  "Prediksi kategori oleh Logistic Regression (TF-IDF, OOF K-Fold)",
    "lr_priority":  "Prediksi prioritas oleh Logistic Regression (TF-IDF, OOF K-Fold)",
    "svm_match":    "TRUE jika SVM benar di category DAN priority sekaligus",
    "rf_match":     "TRUE jika Random Forest benar di category DAN priority sekaligus",
    "lr_match":     "TRUE jika Logistic Regression benar di category DAN priority sekaligus",
    "bert_category": "Prediksi kategori oleh BERT (DistilBERT multilingual, OOF K-Fold)",
    "bert_priority": "Prediksi prioritas oleh BERT (DistilBERT multilingual, OOF K-Fold)",
    "bert_match":   "TRUE jika BERT benar di category DAN priority sekaligus",
}

NOTES_METRICS = {
    "approach":        "Nama model / pendekatan yang dibandingkan (SVM, RF, LR, BERT, Hybrid SVM)",
    "label":           "Label yang dievaluasi: 'category' atau 'priority'",
    "elapsed_seconds": "Total waktu training dan prediksi seluruh fold (dalam detik)",
    "accuracy":        "Akurasi = jumlah prediksi benar / total baris",
    "macro_precision": "Precision rata-rata antar kelas (tidak mempertimbangkan jumlah sampel per kelas)",
    "macro_recall":    "Recall rata-rata antar kelas (tidak mempertimbangkan jumlah sampel per kelas)",
    "macro_f1":        "F1-score rata-rata antar kelas (tidak mempertimbangkan jumlah sampel per kelas)",
    "weighted_f1":     "F1-score rata-rata tertimbang — kelas dengan sampel lebih banyak diberi bobot lebih besar",
    "samples":         "Jumlah total baris yang dievaluasi",
}

NOTES_SUMMARY = {
    "key":   "Nama parameter konfigurasi yang digunakan saat menjalankan notebook",
    "value": "Nilai dari parameter konfigurasi tersebut",
}

def add_header_comments(ws, notes_dict, hybrid_prefix=False):
    for cell in ws[1]:
        col_name = cell.value
        if col_name is None:
            continue
        note = notes_dict.get(col_name)
        if note is None and hybrid_prefix:
            if col_name.startswith("hybrid_svm_category_"):
                model_id = col_name[len("hybrid_svm_category_"):]
                note = f"Prediksi kategori Hybrid SVM: SVM dikoreksi GenAI ({model_id}) untuk baris mismatch"
            elif col_name.startswith("hybrid_svm_priority_"):
                model_id = col_name[len("hybrid_svm_priority_"):]
                note = f"Prediksi prioritas Hybrid SVM: SVM dikoreksi GenAI ({model_id}) untuk baris mismatch"
        if note:
            comment = Comment(note, "System")
            comment.width = 340
            comment.height = 60
            cell.comment = comment

wb = openpyxl.load_workbook(OUTPUT_FILE)
add_header_comments(wb["Predictions_Compare"], NOTES_PREDICTIONS, hybrid_prefix=True)
add_header_comments(wb["Metrics"],             NOTES_METRICS)
add_header_comments(wb["Summary"],             NOTES_SUMMARY)
wb.save(OUTPUT_FILE)

print(f"Output tersimpan: {OUTPUT_FILE} ({len(df)} baris total)")
print("Keterangan header ditambahkan di semua sheet (hover mouse ke nama kolom untuk melihat)")
display(summary_df)


Output tersimpan: ../results/cobacek_compare_20data.xlsx (20 baris total)
Keterangan header ditambahkan di semua sheet (hover mouse ke nama kolom untuk melihat)


,key,value
0,selected_models,"gpt-5.4-mini, gpt-4.1-mini, gpt-4o-mini"
1,bert_model,distilbert-base-multilingual-cased
2,total_rows,20
3,k_folds,5
4,svm_mismatch_rows_corrected,16
